# backward-on-scalar-loss — ex2: vector-output backward() requires gradient= (VJP) — verify x.grad matches y.sum().backward()

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `backward-on-scalar-loss`. Running the final beacon cell reports progress against the `PyTorch: backward()` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: backward()` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backward-on-scalar-loss`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backward-on-scalar-loss"
DD_SUBTOPIC = "PyTorch: backward()"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `backward()` on a non-scalar — the `gradient=` argument (VJP)

Ex1 called `loss.backward()` after reducing per-sample loss to a scalar. The deepening move skips the reduce: call `.backward()` on a VECTOR `y` and pass an explicit `gradient=v` argument. This is the vector-Jacobian product (VJP) — `x.grad = J^T @ v`.

```python
y = f(x)                              # y shape (N,), x shape (N,)
v = torch.ones_like(y)                # this picks the row of J^T
y.backward(gradient=v)                # x.grad = sum over i of dy_i/dx
```

Why this is equivalent to `y.sum().backward()`. Reducing with sum and then differentiating is `d(sum(y))/dx = sum_i dy_i/dx = J^T @ 1`. Passing `gradient=ones_like(y)` is the SAME computation, just spelled out as the VJP directly.

**Why you can't omit `gradient=` for non-scalar `y`.** PyTorch needs to know which scalar function of `y` you are differentiating. For a scalar `y`, the only choice is `y` itself, so `gradient=` defaults to `torch.tensor(1.0)`. For a vector `y`, there's no canonical choice — calling `.backward()` without `gradient=` raises `RuntimeError`.

**The full Jacobian falls out of N VJPs.** Pass `gradient=e_i` (a one-hot vector) and you recover the i-th row of `J^T` — i.e. column of `J`. Stack N such calls and you have the full Jacobian.

### Exercise 2 — vector-output backward() requires gradient= (VJP) — verify x.grad matches y.sum().backward()

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `y.backward(gradient=ones_like(y))` to compute `x.grad = J^T @ 1` on a non-scalar `y`, verify the result matches `y.sum().backward()`, and confirm `y.backward()` without an argument raises `RuntimeError`.
> Keywords: autograd, backward, VJP, gradient
> ```

**KCs targeted:** `non-scalar-backward-needs-gradient-argument`, `vjp-equals-sum-then-backward`

Implement `ex2_vjp_via_backward(x, f)`. Compute the gradient of `sum(f(x))` w.r.t. `x` using the VJP form of `.backward()`.

Inputs:
- `x`: 1-D float tensor (will need `requires_grad=True` internally — see step 1).
- `f`: a callable `f(x) -> Tensor` where `f(x)` has the same shape as `x` (vector → vector).

Algorithm:
1. Make a fresh leaf `x_leaf = x.detach().clone().requires_grad_(True)`. Do NOT modify the caller's `x`.
2. Compute `y = f(x_leaf)`.
3. Verify `y` is NOT a scalar (`y.dim() > 0`). If it IS a scalar, raise `ValueError("f must return a non-scalar tensor")`.
4. Build `v = torch.ones_like(y)`.
5. Call `y.backward(gradient=v)`.
6. Return `x_leaf.grad.detach().clone()`.

Output: 1-D tensor of the same shape as `x` containing the VJP, which equals `d(sum(f(x)))/dx`.

In [ ]:
def ex2_vjp_via_backward(x: Tensor, f) -> Tensor:
    """Compute d(sum(f(x)))/dx via y.backward(gradient=ones_like(y))."""
    raise NotImplementedError()


def _test_ex2():
    # === f(x) = x^2  →  d(sum(x^2))/dx = 2x ===
    x = t.tensor([1.0, 2.0, 3.0, 4.0])
    grad = ex2_vjp_via_backward(x, lambda v: v ** 2)
    expected = 2 * x
    assert t.allclose(grad, expected), f'd(sum(x^2))/dx mismatch: expected={expected}, got {grad}'

    # === Caller's x is unchanged: still a leaf with no grad ===
    assert not x.requires_grad, 'caller x must not have requires_grad flipped on'
    assert x.grad is None, 'caller x must have no .grad'

    # === Matches y.sum().backward() exactly ===
    x2 = t.tensor([0.5, -1.5, 2.0], requires_grad=True)
    y2 = (x2 ** 3 + 2 * x2)
    y2.sum().backward()
    ref = x2.grad.clone()
    x3 = t.tensor([0.5, -1.5, 2.0])
    vjp = ex2_vjp_via_backward(x3, lambda v: v ** 3 + 2 * v)
    assert t.allclose(vjp, ref), f'VJP must equal sum-backward: ref={ref}, got {vjp}'

    # === f(x) = x (identity)  →  grad is all-ones ===
    x = t.tensor([10.0, -3.0, 0.0])
    grad = ex2_vjp_via_backward(x, lambda v: v.clone())
    assert t.allclose(grad, t.ones(3)), f'd(sum(x))/dx must be ones, got {grad}'

    # === Larger function: f(x) = sin(x)  →  grad = cos(x) ===
    x = t.tensor([0.0, 0.1, 1.0, 2.0])
    grad = ex2_vjp_via_backward(x, t.sin)
    assert t.allclose(grad, t.cos(x), atol=1e-6), f'd(sum(sin(x)))/dx must equal cos(x), got {grad}'

    # === f returns a scalar → ValueError ===
    x = t.tensor([1.0, 2.0])
    try:
        ex2_vjp_via_backward(x, lambda v: v.sum())
    except ValueError as e:
        assert 'non-scalar' in str(e).lower() or 'scalar' in str(e).lower(), (
            f'error message must mention scalar, got {e!r}'
        )
    else:
        raise AssertionError('expected ValueError for scalar f(x)')

    # === Output is a detached, leaf-free clone (mutation-safe) ===
    x = t.tensor([1.0, 2.0, 3.0])
    out = ex2_vjp_via_backward(x, lambda v: v ** 2)
    assert not out.requires_grad, 'output must be detached'
    out[0] = 999.0  # should NOT bleed back into any captured leaf
    out2 = ex2_vjp_via_backward(x, lambda v: v ** 2)
    assert out2[0] != 999.0, 'output must be a fresh tensor each call'

    # === Sanity: vector-output without gradient= raises RuntimeError ===
    # This documents WHY ex2 must pass gradient=; not a test of your fn, but of PyTorch.
    x_chk = t.tensor([1.0, 2.0], requires_grad=True)
    y_chk = x_chk ** 2
    try:
        y_chk.backward()
    except RuntimeError:
        pass
    else:
        raise AssertionError(
            'PyTorch should raise RuntimeError on non-scalar backward() without gradient=. '
            'Test invariant broken.'
        )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_vjp_via_backward(x, f):
    x_leaf = x.detach().clone().requires_grad_(True)
    y = f(x_leaf)
    if y.dim() == 0:
        raise ValueError('f must return a non-scalar tensor')
    v = t.ones_like(y)
    y.backward(gradient=v)
    return x_leaf.grad.detach().clone()
```

**`x.detach().clone().requires_grad_(True)` is the safe leaf recipe.** `detach` strips any history, `clone` ensures a fresh storage (so mutation of the leaf doesn't bleed into the caller's view), `requires_grad_(True)` makes it a new computational-graph root.

**VJP with `v=ones` IS sum-then-backward.** The chain rule gives `d(sum_i y_i)/dx_j = sum_i dy_i/dx_j = (J^T @ ones)_j`. Passing `gradient=ones_like(y)` is just the explicit VJP form of the same computation. PyTorch has no "sum first" mode under the hood — `y.sum().backward()` literally inserts a sum node before computing the same VJP.

**Returning a detached clone.** The leaf's `.grad` is accumulator state PyTorch may mutate later (e.g. another `.backward()` call inside the same graph). Returning a detached clone gives the caller a stable snapshot.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()